In [1]:
import torch
import argparse
import os
import cag.dataset as cagds
import cag.similarity as cagsim
from time import time
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer
from transformers.cache_utils import DynamicCache
import logging 
from config import ConfigName, set_config
from threading import Thread

/root/anaconda3/envs/CAG/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
os.environ["CUDA_VISIBLE_DEVICES"]='0'
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found")

In [4]:
global model_name, model, tokenizer
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [5]:
torch.serialization.add_safe_globals([DynamicCache])
torch.serialization.add_safe_globals([set])

In [6]:
def preprocess_knowledge(
                model,
                tokenizer,
                prompt: str,
            ) -> DynamicCache:
    """
    Prepare knowledge kv cache for CAG.
    Args:
        model: HuggingFace model with automatic device mapping
        tokenizer: HuggingFace tokenizer
        prompt: The knowledge to preprocess, which is basically a prompt

    Returns:
        DynamicCache: KV Cache
    """
    print('preprocess_knowledge')
    embed_device = model.model.embed_tokens.weight.device
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(embed_device)
    past_key_values = DynamicCache()
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            past_key_values=past_key_values,
            use_cache=True,
            output_attentions=False,
            output_hidden_states=False
        )
    print('complete')
    return outputs.past_key_values

In [7]:
def write_kv_cache(kv: DynamicCache, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    """
    Write the KV Cache to a file.
    """
    print('write_kv_cache')
    torch.save(kv, path)

In [8]:
def load_quantized_model(model_name, hf_token=None):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        token=hf_token
    )
    print('load_quantized_model')
    # Load model with quantization
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        # device_map='auto',          # Automatically choose best device
        trust_remote_code=True,     # Required for some models
        token=hf_token
    ).to(device)

    return tokenizer, model

In [9]:
def prepare_kvcache(documents, filepath: str = "./data_cache/cache_knowledges.pt", answer_instruction: str | None = None):
    # Prepare the knowledges kvcache
    # print('prepare_kvcache')

    if answer_instruction is None:
        answer_instruction = "請簡短回答。"#"Answer the question with a super short answer."
    # knowledges = f"""
    # <|begin_of_text|>
    # <|start_header_id|>system<|end_header_id|>
    # You are an assistant for giving short answers based on given context.<|eot_id|>
    # <|start_header_id|>user<|end_header_id|>
    # Context information is bellow.
    # ------------------------------------------------
    # {documents}
    # ------------------------------------------------
    # {answer_instruction}
    # Question:
    # """
    
    knowledges = f"""
    <|begin_of_text|>
    <|start_header_id|>system<|end_header_id|>
    以你的理解並根據情況判斷是否使用參考資訊後回答問題，但是回答不能有偽造成分。請用繁體中文回答問題，保持回答生動。<|eot_id|>
    <|start_header_id|>user<|end_header_id|>
    上下文資訊如下。
    ------------------------------------------------
    {documents}
    ------------------------------------------------
    問題:
    """
    
    # Get the knowledge cache
    t1 = time()
    kv = preprocess_knowledge(model, tokenizer, knowledges)
    print("kvlen: ", kv.key_cache[0].shape[-2])
    write_kv_cache(kv, filepath)
    t2 = time()
    # logger.info(f"KV cache prepared in {t2 - t1:.2f} seconds.")
    return kv, t2 - t1

In [10]:
# Define quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # Load model in 4-bit precision
    bnb_4bit_quant_type="nf4",      # Normalize float 4 quantization
    bnb_4bit_compute_dtype=torch.bfloat16,  # Compute dtype for 4-bit base matrices
    bnb_4bit_use_double_quant=True  # Use nested quantization
)

model_name = '../../Chinese-LLaMA-Alpaca-3/model/llama-3-chinese-8b-instruct-v3' # Llama-3.1-8B-Instruct
kvcache = 'file'
maxQuestion = 10
maxParagraph = 10
maxKnowledge = 5
usePrompt = False
dataset = './datasets/Long-term_care/長照文章_2.txt'
randomSeed = 0
quantized = True

set_config(ConfigName.RAND_SEED, randomSeed)

#load llm model
tokenizer, model = load_quantized_model(model_name=model_name, hf_token=HF_TOKEN)
# load llm model
# tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype=torch.float16,
#     device_map="auto",
#     token=HF_TOKEN
# )

answer_instruction =  "請簡短回答。"
#text_list, dataset = cagds.get(dataset, max_knowledge=maxKnowledge,
#                               max_paragraph=maxParagraph, 
#                               max_questions=maxQuestion)

#print(text_list)
kvcache_path = "./data_cache/long-term_care_knowledges_2.pt"

#knowledges = '\n\n\n\n\n\n'.join(text_list)
knowledges = []
with open(dataset, 'r', encoding='utf-8') as dataset_read:
    knowledges = dataset_read.readlines()
    #knowledges.append()

knowledge_cache, prepare_time = prepare_kvcache(knowledges, filepath=kvcache_path,
                                                answer_instruction=answer_instruction)
kv_len = knowledge_cache.key_cache[0].shape[-2]
#dataset = list(dataset)  # Convert the dataset to a list



load_quantized_model


`low_cpu_mem_usage` was None, now default to True since model is quantized.
Loading checkpoint shards: 100%|██████████████████| 4/4 [02:31<00:00, 37.91s/it]


preprocess_knowledge
complete
kvlen:  6810
write_kv_cache


In [37]:
from llama_index.core import VectorStoreIndex, Document
def getBM25Retriever(documents: list[Document], similarity_top_k: int=3,tokenizer=tokenizer):
    from llama_index.core.node_parser import SentenceSplitter  
    from llama_index.retrievers.bm25 import BM25Retriever
    import Stemmer

    splitter = SentenceSplitter(chunk_size=512)
    
    t1 = time()
    nodes = splitter.get_nodes_from_documents(documents)
    # We can pass in the index, docstore, or list of nodes to create the retriever
    bm25_retriever = BM25Retriever.from_defaults(
        nodes=nodes,
        similarity_top_k=similarity_top_k,
        tokenizer=tokenizer,
        # stemmer=Stemmer.Stemmer("english"),
        # language="english",
    )
    t2 = time()
    bm25_retriever.persist("./retriver/bm25_retriever")

    return bm25_retriever, t2 - t1
document = [Document(text=t) for t in knowledges]
# print(document)
# retriever, prepare_time = getBM25Retriever(document, similarity_top_k=3)

In [12]:
def clean_up(kv: DynamicCache, origin_len: int):
    """
    Truncate the KV Cache to the original length.
    """
    for i in range(len(kv.key_cache)):
        kv.key_cache[i] = kv.key_cache[i][:, :, :origin_len, :]
        kv.value_cache[i] = kv.value_cache[i][:, :, :origin_len, :]

In [13]:
def generate(
            model,
            input_ids: torch.Tensor,
            past_key_values,
            max_new_tokens: int = 300
        ) -> torch.Tensor:
    """
    Generate text with greedy decoding.

    Args:
        model: HuggingFace model with automatic device mapping
        input_ids: Input token ids
        past_key_values: KV Cache for knowledge
        max_new_tokens: Maximum new tokens to generate
    """
    # print('generate')
    embed_device = model.model.embed_tokens.weight.device
    
    if past_key_values is None:
        new_prompt = f"""
    <|begin_of_text|>
    <|start_header_id|>system<|end_header_id|>
    以你的理解回答問題，但是回答不能有偽造成分。請用繁體中文回答問題，保持回答簡潔生動。<|eot_id|>
    <|start_header_id|>user<|end_header_id|>
    問題:
    """
        input_ids = tokenizer.encode(new_prompt + prompt, return_tensors="pt").to(model.device)
    origin_ids = input_ids
    input_ids = input_ids.to(embed_device)

    output_ids = input_ids.clone()
    next_token = input_ids
    # print(past_key_values)
    st_time = time()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(
                input_ids=next_token, 
                past_key_values=past_key_values,
                # streamer=streamer,
                use_cache=True
            )
            next_token_logits = outputs.logits[:, -1, :]
            next_token = next_token_logits.argmax(dim=-1).unsqueeze(-1)
            next_token = next_token.to(embed_device)

            past_key_values = outputs.past_key_values

            output_ids = torch.cat([output_ids, next_token], dim=1)
            # print(tokenizer.decode(output_ids[:, origin_ids.shape[-1]:][0], 
            #                           skip_special_tokens=True, 
            #                           temperature=None))
            # yield tokenizer.decode(output_ids[:, origin_ids.shape[-1]:][0], 
            #                           skip_special_tokens=True, 
            #                           temperature=None)
            # for new_text in streamer:
            #     print(new_text)
            if next_token.item() in [model.config.eos_token_id]:
                break
        # print(output_ids)
    # print(time()-st_time)
    return output_ids[:, origin_ids.shape[-1]:]

In [13]:
question = "請介紹YunTech one電動車"
prompt = f"""
{question}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""
clean_up(knowledge_cache, kv_len)
embed_device = model.model.embed_tokens.weight.device
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
cag_generate_t = time()
output = generate(model, input_ids, knowledge_cache)
cag_generated_text = tokenizer.decode(output[0], 
                                      skip_special_tokens=True, 
                                      temperature=None)
cag_generate_t = time() - cag_generate_t

DynamicCache()
14.04084300994873


In [16]:
clean_up(knowledge_cache, kv_len)
embed_device = model.model.embed_tokens.weight.device
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
generate_t = time()
output = generate(model, input_ids, None)
none_generated_text = tokenizer.decode(output[0],
                                       skip_special_tokens=True,
                                       temperature=None)
generate_t = time()-generate_t

We detected that you are passing `past_key_values` as a tuple of tuples. This is deprecated and will be removed in v4.47. Please convert your cache or use an appropriate `Cache` class (https://huggingface.co/docs/transformers/kv_cache#legacy-cache-format)


None
10.112877130508423


In [17]:
rag_retrieve_t = time()
nodes = retriever.retrieve(question)
rag_retrieve_t = time()-rag_retrieve_t

In [18]:
knowledge = "\n---------------------\n".join([node.text for node in nodes])

In [19]:
rag_prompt = f"""
    <|begin_of_text|>
    <|start_header_id|>system<|end_header_id|>
    You are an assistant for giving short answers based on given context.<|eot_id|>
    <|start_header_id|>user<|end_header_id|>
    Context information is below.
    ------------------------------------------------
    {knowledge}
    ------------------------------------------------
    {answer_instruction}
    Question:
    {question}
    <|eot_id|>
    <|start_header_id|>assistant<|end_header_id|>
    """

In [20]:
input_ids = tokenizer.encode(rag_prompt, return_tensors="pt").to(model.device)

In [24]:
rag_gererate_t = time()
output = model.generate(
            input_ids,
            max_new_tokens=500,  # Set the maximum length of the generated text
            do_sample=False,  # Ensures greedy decoding,
            temperature=None
        )
rag_gererate_t = (time()-rag_gererate_t)+rag_retrieve_t
rag_generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
rag_generated_text = rag_generated_text[rag_generated_text.find(question) + len(question):]
rag_generated_text = rag_generated_text[rag_generated_text.find('assistant') + len('assistant'):].lstrip()

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


In [22]:
none_generated_text,generate_t

('YunTech one是一款由台灣雲端科技股份有限公司所研發的電動車，於2019年正式上市。這款電動車採用了鋰電池技術，具有高效能、低噪音、低排放等優點。YunTech one的外觀設計簡潔大方，車身採用了輕量化的鋁合金材質，車身重量僅為1,200公斤，具有良好的操控性能。車身長度為4,390mm，寬度為1,780mm，高度為1,620mm，車身尺寸適中，適合城市道路行駛。\nYunTech one的最大特點是其高效能的電動機動力系統，最大功率達到110kW，最大扭力達到250Nm，具有強大的加速性能。車輛最高時速可達160km/h，續航里程達到300km以上，充電時間僅需約2.5小時。YunTech one還配備了多項智能科技，包括自動泊車、自動巡航、車道保持等功能，提供了更便捷的駕駛體驗。\n總體來說，YunTech one是一款性能優異、環保節能的電動車，適合城市通勤',
 10.11443018913269)

In [14]:
cag_generated_text, cag_generate_t

('YunTech ONE是一款智慧電動概念車，由國立雲林科技大學與e-Team聯盟廠商共同研發，展現了雲科大產學聯盟與跨設計、管理、工程三學院之整合能力。車體外型、內裝設計及採用無方向盤全自駕技術，展現智慧電動車技術的成果。雲科大致力於智慧座艙和自駕車技術的研發，並透過國際車廠的測試和認證，提供智慧電動車產業所需的人才和技術支援。',
 14.042026281356812)

In [26]:
rag_generated_text, rag_gererate_t

('YunTech ONE是一個智慧電動車產業發表會，旨在探討智慧電動車體系的現況、機會和挑戰，並促進產官研界的交流與合作。它由雲科大和e-Team聯盟共同舉辦，旨在推動台灣智慧電動車產業的發展。',
 10.23974871635437)

In [27]:
knowledge

'「YunTech ONE」發表會有眾多產、官、學、研界重量級貴賓與會參加，總統府的林信義資政以「探討智慧電動車體系的現況、機會和挑戰」發表精彩專題演講。會場有智慧電動車技術、製造、測試等領域貴賓及產業界進行互動，進一步促進密切的交流與合作。楊能舒強調，藉由「YunTech ONE」發表會不僅展現雲科大與e-Team聯盟研發成果，針對台灣電動車產業未來的策略與努力方向，成功為推動台灣智慧電動車產業再向前邁出關鍵一步。\n---------------------\n值得一提，應全球智慧電動車產業需求與發展趨勢，雲科大2010年起投入汽車領域技術研發，2019年進一步成立e-Team聯盟，啟動智慧電動車研發，雲林縣政府與雲林科技大學合作，規畫在雲科大校區南側至龍潭南路以西、面積約328公頃腹地，設置「斗六智慧電動車產業園區」，以打造雲林電動車產業專區，第一期約234公頃，扣除公共設施後，產業用地約83公頃，第二期則約94公頃。目前由雲林縣都市計畫委員大會專案小組審查中，雲林縣府表示，通過政策環評後，預定最快明年上半年送交內政部審查，同時啟動都市計畫徵收。\n---------------------\n「YunTech ONE」發表會同時，舉辦2023TEGA論壇「智慧電動車時代台灣產業之挑戰與雲林之契機」，由TEGA協會理事長車蔡裕慶董事長擔任論壇主持，宣明智董事長擔任論壇引言人。並邀請雲林縣政府張麗善縣長、慧國工業江瑞坤總經理、車輛研究測試中心廖學隆副總以及雲科大蘇純繒副校長等產官研及車業領袖參與論壇。分享他們在汽車產業方面的經驗和策略，以指導雲林和台灣汽車產業的發展戰略和佈局，並冀望協助雲林在汽車產業以及雲林智慧電動車產業園區之設立，能獲得更多的發展與成效。'

In [12]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # Load model in 4-bit precision
    bnb_4bit_quant_type="nf4",      # Normalize float 4 quantization
    bnb_4bit_compute_dtype=torch.bfloat16,  # Compute dtype for 4-bit base matrices
    bnb_4bit_use_double_quant=True  # Use nested quantization
)

model_name = '../../Chinese-LLaMA-Alpaca-3/model/llama-3-chinese-8b-instruct-v3' # Llama-3.1-8B-Instruct
kvcache = 'file'
maxQuestion = 10
maxParagraph = 10
maxKnowledge = 5
usePrompt = False
# dataset = './datasets/yuntech_one/雲科電動車_240128.txt'
randomSeed = 0
quantized = True

set_config(ConfigName.RAND_SEED, randomSeed)

#load llm model
tokenizer, model = load_quantized_model(model_name=model_name, hf_token=HF_TOKEN)

# tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype=torch.float16,
#     device_map="auto",
#     token=HF_TOKEN
# )

load_quantized_model


`low_cpu_mem_usage` was None, now default to True since model is quantized.
Loading checkpoint shards: 100%|██████████████████| 4/4 [00:05<00:00,  1.46s/it]


In [19]:
kv = torch.load('./data_cache/long-term_care_knowledges_2.pt', map_location='cuda:0')

/tmp/ipykernel_29043/3698187798.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  kv = torch.load('./data_cache/long-term_care_knowledges_2.pt', map_location='cuda:0')


In [23]:
question = "我們想申請喘息服務或輔具，有哪些政府長照資源可以用？"#回答請控制在100字內。"
answer_prompt = '回答請控制在100字內。'
prompt = f"""
{question}\n{answer_prompt}<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""
kv_len = kv.key_cache[0].shape[-2]
clean_up(kv, kv_len)
embed_device = model.model.embed_tokens.weight.device
streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
cag_generate_t = time()
#generation_kwargs = dict(model = model,streamer=streamer, input_ids = input_ids, past_key_values=kv)
output = generate(model, input_ids, kv)
# thread = Thread(target=generate, kwargs=generation_kwargs)
# for new_text in streamer:
#     print(new_text)
for new_text in output:
    aaaa = new_text
cag_generated_text = tokenizer.decode(aaaa, 
                                      skip_special_tokens=True, 
                                      temperature=None)
print(cag_generated_text)
cag_generate_t = time() - cag_generate_t

您可以申請長照服務或輔具的資源，包括居家照顧、日間照顧、居家服務、喘息服務、輔具租借等。您可以聯繫當地的長照服務中心或社區長照中心，詢問相關資訊和申請流程。另外，您也可以參考內政部長照署的網站，了解更多長照資源和服務的資訊。


In [52]:
cag_generated_text, cag_generate_t

('YunTech ONE是雲林科技大學與e-Team聯盟共同研發的智慧電動概念車，展現了雲科大產學聯盟與跨設計、管理、工程三學院之整合能力。車體外型、內裝設計及採用無方向盤全自駕技術，展現智慧電動車技術的成果。雲科大致力於智慧座艙和自駕車技術的研發，並透過國際車廠的測試和認證，提供智慧電動車產業所需的人才和技術支援。',
 14.545490503311157)

In [14]:
questions = ["我們家照顧長輩主要靠媽媽一個人，這樣會不會太辛苦？有什麼方式可以分擔？",
           "如果我們家有長輩失智了，要怎麼學會正確又不太累的照顧方法？",
           "家裡老人最近常說睡不好、吃不下，我們要不要帶去看醫生？會是什麼警訊？",
           "長輩有主動脈瓣膜狹窄，醫師建議做TAVI手術，這個手術安全嗎？恢復時間長嗎？",
           "我們兄弟姊妹各有家庭，怎麼分工照顧爸媽比較公平？有實際建議嗎？",
           "爸媽年紀大了，萬一以後失能或住院，我們該怎麼提前做好規劃？",
           "聽說「老老照顧」很辛苦，如果兩個老人互相照顧，要注意什麼？",
           "爸媽有假牙，有時吃不太下，要不要定期檢查口腔？有補助嗎？",
           "我們想申請喘息服務或輔具，有哪些政府長照資源可以用？",
           "有什麼食物能幫助老人預防失智？平常可以怎麼吃比較好？",]#回答請控制在100字內。"
answer_instruction = '回答請控制在100字內。'
output_dir = './output'

In [15]:
num_of_top_k = [3, 5, 7]
for top_k in num_of_top_k:
    avg_rag_gen_time = 0
    retriever, prepare_time = getBM25Retriever(document, similarity_top_k=top_k, tokenizer=tokenizer)
    for question in questions:
        rag_retrieve_t = time()
        nodes = retriever.retrieve(question)
        rag_retrieve_t = time()-rag_retrieve_t

        knowledge = "\n---------------------\n".join([node.text for node in nodes])
        rag_prompt = f"""
            <|begin_of_text|>
            <|start_header_id|>system<|end_header_id|>
            以你的理解並根據情況判斷是否使用參考資訊後回答問題，但是回答不能有偽造成分。請用繁體中文回答問題，保持回答生動。<|eot_id|>
            <|start_header_id|>user<|end_header_id|>
            上下文資訊如下。
            ------------------------------------------------
            {knowledge}
            ------------------------------------------------
            {answer_instruction}
            問題:
            {question}
            <|eot_id|>
            <|start_header_id|>assistant<|end_header_id|>
            """
        input_ids = tokenizer.encode(rag_prompt, return_tensors="pt").to(model.device)
        rag_gererate_t = time()
        output = model.generate(
                    input_ids,
                    max_new_tokens=500,  # Set the maximum length of the generated text
                    do_sample=False,  # Ensures greedy decoding,
                    temperature=None
                )
        rag_gererate_t = (time()-rag_gererate_t)+rag_retrieve_t
        rag_generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
        rag_generated_text = rag_generated_text[rag_generated_text.find(question) + len(question):]
        rag_generated_text = rag_generated_text[rag_generated_text.find('assistant') + len('assistant'):].lstrip()
        avg_rag_gen_time += rag_gererate_t
        with open(os.path.join(output_dir, f'retrive_nodes_top{top_k}.txt'), 'a', encoding='utf-8') as f:
            f.write(f"問題:\n{question}\n")
            f.write(f"檢索:\n{knowledge}\n\n")
            f.write('='*20 + '\n')
        with open(os.path.join(output_dir, f'retrive_answer_top{top_k}.txt'), 'a', encoding='utf-8') as f:
            f.write("給定一個問題和回答,請從給定的答案中提取一個或多個敘述。\n")
            f.write(f"問題:\n{question}\n\n")
            f.write(f"回答:\n{rag_generated_text}\n\n")
            f.write('='*20 + '\n')

Finding newlines for mmindex: 100%|███████████| 128k/128k [00:00<00:00, 108MB/s]
/root/anaconda3/envs/CAG/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your inpu

In [25]:
avg_cache_time = 0
kv = torch.load('./data_cache/long-term_care_knowledges_2.pt', map_location='cuda:0')
kv_len = kv.key_cache[0].shape[-2]
for question in questions:
    prompt = f"""
    {question}\n{answer_instruction}<|eot_id|>
    <|start_header_id|>assistant<|end_header_id|>
    """
    clean_up(kv, kv_len)
    embed_device = model.model.embed_tokens.weight.device
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    cag_generate_t = time()
    #generation_kwargs = dict(model = model,streamer=streamer, input_ids = input_ids, past_key_values=kv)
    output = generate(model, input_ids, kv, max_new_tokens = 500)
    # thread = Thread(target=generate, kwargs=generation_kwargs)
    # for new_text in streamer:
    #     print(new_text)
    # for new_text in output:
    #     cag_generated_text = new_text
    cag_generated_text = tokenizer.decode(output[0], 
                                          skip_special_tokens=True, 
                                          temperature=None)
    with open(os.path.join(output_dir, f'cache_answer.txt'), 'a', encoding='utf-8') as f:
        f.write("給定一個問題和回答,請從給定的答案中提取一個或多個敘述。\n")
        f.write(f"問題:\n{question}\n\n")
        f.write(f"回答:\n{cag_generated_text}\n\n")
        f.write('='*20 + '\n')
    cag_generate_t = time() - cag_generate_t
    avg_cache_time += cag_generate_t

/tmp/ipykernel_29043/3671936365.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  kv = torch.load('./data_cache/long-term_care_knowledges_2.pt', map_location='cuda:0')


In [17]:
with open(os.path.join(output_dir, f'avg_gen_time.txt'), 'a', encoding='utf-8') as f:
    f.write(f"rag_avg_time : {avg_rag_gen_time/len(questions)}\n")
    f.write(f"cag_avg_time : {avg_cache_time/len(questions)}\n")

In [39]:
retriever.retrieve("我們家照顧長輩主要靠媽媽一個人，這樣會不會太辛苦？有什麼方式可以分擔？")

[NodeWithScore(node=TextNode(id_='4ee1bf80-0962-4169-876f-3e5ea1c088f1', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='088ebf3b-e067-438a-b3df-e1b053cdbf08', node_type='4', metadata={}, hash='60fb6714851646f59d52d2463c5e7381fa429b003cac6d64f43b64ff8af9ef64')}, metadata_template='{key}: {value}', metadata_separator='\n', text='黃啟宏醫師也提醒，「經導管主動脈瓣膜植入術」成功率雖然高，但少部分患者可能在手術中產生心臟突然不跳、中風等風險。治療時，需要專業內、外科醫師共同合作、執行，治療才會精準。另外，並非所有患者都適合這種手術，必須經醫師專業評估，以擬訂適合治療策略。', mimetype='text/plain', start_char_idx=0, end_char_idx=122, metadata_seperator='\n', text_template='{metadata_str}\n\n{content}'), score=0.0),
 NodeWithScore(node=TextNode(id_='5cd2f63d-a160-4dad-8b98-b5323aedd920', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='fed4d66e-4bf4-4c65-8ffa-f8688fa480d7', node_type

In [38]:
retriever, prepare_time = getBM25Retriever(document, similarity_top_k=7, tokenizer=tokenizer)

The tokenizer parameter is deprecated and will be removed in a future release. Use a stemmer from PyStemmer instead.
Finding newlines for mmindex: 100%|███████████| 128k/128k [00:00<00:00, 150MB/s]
